# Exploratory Data Analysis (EDA): Mental Health NLP & Twitter Network Topology

**Consulting Engagement: BigDataCompany - Big Data Analytics Architecture**  
*Course: Big Data Analysis (BSc Labs) - NOVA IMS*  
*Client Prototype: Integrated Mental Health & Social Network Intelligence Platform*

---

### Executive Overview & The 4 V's Assessment
To provide our client with actionable data intelligence, this notebook implements distributed exploratory data profiling using **Apache Spark**. Rather than relying on single-node tools (like pandas) which hit severe memory and I/O bottlenecks when processing large social graphs, PySpark allows us to evaluate the full scope of the client's data across the **4 V's**:

1. **Volume:** Scalable analysis across **548,541 directed interaction edges** and **42,518 clinical/social text records**.
2. **Velocity:** Parsing UTC relationship timestamps to track conversational dynamics and prepare for real-time streaming ingestion.
3. **Variety:** Processing multi-modal data structures—combining **unstructured natural language text** with **relational graph topology**.
4. **Veracity:** Establishing data reliability by auditing missing values with distributed aggregations, inspecting schema types, and detecting anomalies.


In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, 
    DoubleType, BooleanType, TimestampType
)
from pyspark.sql.functions import (
    col, count, when, isnan, length, size, split, lower, 
    trim, desc, asc, sum as spark_sum, avg as spark_avg, 
    min as spark_min, max as spark_max, stddev as spark_stddev,
    explode, to_timestamp, to_date, date_trunc, row_number, dense_rank
)
from pyspark.sql.window import Window

# Initialize Spark Session with optimized cluster settings
spark = SparkSession.builder \
    .appName("EDA_MentalHealth_SocialGraph") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

# Flexible path resolution for Databricks DBFS and Local Repositories
def resolve_data_path(filename):
    candidates = [
        f"/FileStore/tables/{filename}",
        f"dbfs:/FileStore/{filename}",
        f"../Datasets/{filename}",
        f"Datasets/{filename}",
        f"../{filename}",
        filename
    ]
    for p in candidates:
        clean_p = p.replace("dbfs:", "")
        if os.path.exists(clean_p) or p.startswith("/FileStore") or p.startswith("dbfs:"):
            return p
    return f"Datasets/{filename}"

mental_health_path = resolve_data_path("mental_health.csv")
twitter_graph_path = resolve_data_path("twitter_graph.csv")

print(f"Resolved Mental Health Path: {mental_health_path}")
print(f"Resolved Twitter Graph Path:  {twitter_graph_path}")


## 1. Enterprise Schema Architecture (Explicit StructType)

In distributed big data architectures, relying on `inferSchema=True` is an anti-pattern:
* It forces Spark to execute an entire **extra distributed scan** over large files before processing begins.
* It risks parsing large 64-bit Twitter IDs as floating-point numbers, causing catastrophic precision loss and scientific notation distortion.

Here we explicitly define the schema upfront using `StructType` and `StructField`.


In [ ]:
# Explicit Schema for Mental Health Text Data
mental_health_schema = StructType([
    StructField("ID", LongType(), nullable=False),
    StructField("Text", StringType(), nullable=True),
    StructField("Emotion", StringType(), nullable=True)
])

# Explicit Schema for Twitter Interaction Graph
# Notice: Source and Destination IDs are strictly typed as StringType to prevent precision loss!
twitter_graph_schema = StructType([
    StructField("Source", StringType(), nullable=True),
    StructField("Destination", StringType(), nullable=True),
    StructField("Conversation ID", StringType(), nullable=True),
    StructField("Relationship Date (UTC)", StringType(), nullable=True),
    StructField("Source Joined Date", StringType(), nullable=True),
    StructField("Source Followers", DoubleType(), nullable=True),
    StructField("Source Followed", DoubleType(), nullable=True),
    StructField("Source Verified", StringType(), nullable=True),
    StructField("Destination Joined Date", StringType(), nullable=True),
    StructField("Destination Followers", DoubleType(), nullable=True),
    StructField("Destination Followed", DoubleType(), nullable=True),
    StructField("Destination Verified", StringType(), nullable=True)
])

# Read datasets using explicit schemas
mental_health_df = spark.read.csv(
    mental_health_path, 
    header=True, 
    schema=mental_health_schema, 
    multiLine=True, 
    escape='"'
)

twitter_graph_df = spark.read.csv(
    twitter_graph_path, 
    header=True, 
    schema=twitter_graph_schema
)

print("✅ Both datasets ingested successfully with explicit schemas.")
mental_health_df.printSchema()
twitter_graph_df.printSchema()


## 2. Data Veracity: Corrected Null & Missing Value Profiling

### Addressing the Spark Aggregation Bug
In standard SQL / PySpark, `count(col)` only counts **non-null** values. Using `count(when(col.isNull(), col))` is a common trap: when `col` is null, the `when()` statement yields `null`, and `count(null)` evaluates to `0`! 

To accurately count missing values across the entire distributed cluster without collecting rows to the driver, we use:
$$\sum \mathbb{I}(\text{col is null or NaN})$$
implemented via `spark_sum(when(col.isNull() | isnan(col), 1).otherwise(0))` for numerics, and `spark_sum(when(col.isNull(), 1).otherwise(0))` across all types.


In [ ]:
def profile_missing_values(df, dataset_name):
    print(f"=== Missing Value Audit: {dataset_name} ===")
    total_records = df.count()
    
    numeric_dtypes = {"int", "bigint", "double", "float"}
    missing_exprs = []
    for c, dtype in df.dtypes:
        if dtype in numeric_dtypes:
            cond = col(f"`{c}`").isNull() | isnan(col(f"`{c}`"))
        else:
            cond = col(f"`{c}`").isNull()
        missing_exprs.append(spark_sum(when(cond, 1).otherwise(0)).alias(c))
        
    missing_counts = df.select(missing_exprs).collect()[0].asDict()
    
    summary_data = []
    for column, null_cnt in missing_counts.items():
        pct = (null_cnt / total_records) * 100.0 if total_records > 0 else 0.0
        summary_data.append({"Column": column, "Null_Count": null_cnt, "Null_Percentage": round(pct, 3)})
        
    summary_df = pd.DataFrame(summary_data)
    print(f"Total Rows Analyzed: {total_records:,}")
    return summary_df

# Audit Mental Health Dataset
mh_null_summary = profile_missing_values(mental_health_df, "Mental Health Dataset")
print(mh_null_summary.to_string(index=False))

print("\n")
# Audit Twitter Graph Dataset
tg_null_summary = profile_missing_values(twitter_graph_df, "Twitter Interaction Graph")
print(tg_null_summary.to_string(index=False))


## 3. Text Mining Exploratory Analysis

In this section, we analyze the structural and lexical characteristics of the social media text records:
1. **Target Class Balance:** Frequency and percentage distribution of the 6 emotional states.
2. **Document Complexity:** Character length and token/word count distributions.
3. **Lexical Profiling:** Top frequent words per emotion category to evaluate domain vocabulary.


In [ ]:
# 1. Target Class Distribution
total_texts = mental_health_df.count()
emotion_distribution = mental_health_df.groupBy("Emotion") \
    .agg(count("*").alias("Volume")) \
    .withColumn("Percentage", (col("Volume") / total_texts) * 100.0) \
    .orderBy(desc("Volume"))

print("=== Emotion Class Distribution ===")
emotion_distribution.show()

# Collect small aggregated result to driver for visualization
emotion_pd = emotion_distribution.toPandas()

plt.figure(figsize=(10, 5))
palette = sns.color_palette("mako", len(emotion_pd))
bars = plt.bar(emotion_pd["Emotion"], emotion_pd["Volume"], color=palette)
plt.title("Volume of Records by Emotional State (N = 42,518)", fontsize=14, fontweight="bold")
plt.xlabel("Psychological / Emotional Category", fontsize=12)
plt.ylabel("Number of Texts", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.7)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 100, f"{int(yval):,} ({yval/total_texts*100:.1f}%)", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
# 2. Text Complexity Profiling (Word Counts and Character Lengths)
text_metrics_df = mental_health_df \
    .withColumn("Char_Length", length(col("Text"))) \
    .withColumn("Word_Count", size(split(trim(col("Text")), r"\s+")))

print("=== Text Document Complexity Statistics (Overall) ===")
text_metrics_df.select("Char_Length", "Word_Count") \
    .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max") \
    .show()

print("=== Complexity Breakdown by Emotional Category ===")
complexity_by_emotion = text_metrics_df.groupBy("Emotion").agg(
    count("*").alias("Count"),
    spark_avg("Word_Count").alias("Avg_Words"),
    spark_stddev("Word_Count").alias("Std_Words"),
    spark_avg("Char_Length").alias("Avg_Chars"),
    spark_max("Word_Count").alias("Max_Words")
).orderBy(desc("Avg_Words"))

complexity_by_emotion.show()


In [ ]:
# 3. Lexical Profiling: Top Keywords per Emotion
standard_stopwords = {
    "i", "me", "my", "myself", "we", "our", "ours", "you", "your", "he", "him", 
    "his", "she", "her", "it", "its", "they", "them", "what", "which", "who", 
    "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", 
    "be", "been", "being", "have", "has", "had", "having", "do", "does", "did", 
    "doing", "a", "an", "the", "and", "but", "if", "or", "because", "as", "until", 
    "while", "of", "at", "by", "for", "with", "about", "against", "between", "into", 
    "through", "during", "before", "after", "above", "below", "to", "from", "up", 
    "down", "in", "out", "on", "off", "over", "under", "again", "further", "then", 
    "once", "here", "there", "when", "where", "why", "how", "all", "any", "both", 
    "each", "few", "more", "most", "other", "some", "such", "no", "nor", "not", 
    "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will", 
    "just", "don", "should", "now", "feel", "feeling", "im", "like"
}

# Explode words and filter noise
words_df = mental_health_df.select(
    col("Emotion"),
    explode(split(lower(col("Text")), r"[^a-zA-Z]+")).alias("word")
).filter(
    (length(col("word")) > 2) & (~col("word").isin(list(standard_stopwords)))
)

# Window specification to pick top 5 words per emotion
word_counts = words_df.groupBy("Emotion", "word").count()
window_spec = Window.partitionBy("Emotion").orderBy(desc("count"))

top_words_per_emotion = word_counts \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 5) \
    .orderBy("Emotion", "rank")

print("=== Top 5 Defining Vocabulary Terms per Emotional Category ===")
top_words_per_emotion.show(30, truncate=False)


## 4. Social Network Graph Topology Profiling

The Twitter interaction network captures directed information flow between accounts. Understanding the underlying structural properties is essential before building GraphFrames algorithms:
1. **Network Scale:** Vertices count, edge count, density proxy.
2. **Degree Centrality:** In-degree (inbound attention/replies) vs. Out-degree (outbound broadcast activity).
3. **Power-Law Distribution:** Evaluating degree skewness characteristic of scale-free social graphs.
4. **Influential Hubs & Verification:** Correlating follower counts and account verification status.


In [ ]:
# 1. Network Scale
total_edges = twitter_graph_df.count()
unique_sources = twitter_graph_df.select("Source").dropna().distinct().count()
unique_destinations = twitter_graph_df.select("Destination").dropna().distinct().count()

# Total unique entities across both endpoints
all_nodes = twitter_graph_df.select(col("Source").alias("node")) \
    .union(twitter_graph_df.select(col("Destination").alias("node"))) \
    .dropna().distinct().count()

print("=== Twitter Network Topological Scale ===")
print(f"Total Directed Edges:        {total_edges:,}")
print(f"Unique Broadcasting Sources: {unique_sources:,}")
print(f"Unique Target Destinations:  {unique_destinations:,}")
print(f"Total Unique Nodes (Users):  {all_nodes:,}")


In [ ]:
# 2. Out-Degree Distribution (Sources)
out_degree_df = twitter_graph_df.groupBy("Source").agg(count("*").alias("out_degree"))

# 3. In-Degree Distribution (Destinations)
in_degree_df = twitter_graph_df.groupBy("Destination").agg(count("*").alias("in_degree"))

print("=== Out-Degree Distribution Summary (Broadcasting Activity) ===")
out_degree_df.select("out_degree").summary("count", "mean", "stddev", "min", "50%", "75%", "95%", "max").show()

print("=== In-Degree Distribution Summary (Popularity / Target Centrality) ===")
in_degree_df.select("in_degree").summary("count", "mean", "stddev", "min", "50%", "75%", "95%", "max").show()


In [ ]:
# 4. Top Hub Accounts in the Network
print("=== Top 5 In-Degree Target Hubs (Most Mentioned / Interacted) ===")
in_degree_df.orderBy(desc("in_degree")).show(5)

print("=== Top 5 Verified Broadcasting Accounts by Follower Count ===")
twitter_graph_df.filter(col("`Source Verified`") == "True") \
    .select("Source", "`Source Followers`", "`Source Followed`") \
    .dropDuplicates(["Source"]) \
    .orderBy(desc("`Source Followers`")) \
    .show(5)


## 5. Advanced Spark SQL & Analytical Window Functions

To satisfy the course requirement for **Spark SQL queries and distributed Window Functions** (Slide 41), we register both datasets as temporary in-memory views and answer two strategic questions:
1. **Text Query:** Which are the top 3 most expressive (longest) social media statements within each emotional class?
2. **Network Query:** Who are the top 5 destination accounts receiving the highest inbound interactions, partitioned by whether they are verified?


In [ ]:
# Register temporary views for Spark SQL execution
text_metrics_df.createOrReplaceTempView("mental_health_eda")
twitter_graph_df.createOrReplaceTempView("twitter_graph_eda")

print("✅ Spark SQL Temporary Views Registered.")

# Spark SQL Query 1: Top 3 Most Expressive Posts per Emotion Class using DENSE_RANK()
query_text_window = """
WITH RankedPosts AS (
    SELECT 
        ID,
        Emotion,
        Word_Count,
        Char_Length,
        SUBSTRING(Text, 1, 80) AS Snippet,
        DENSE_RANK() OVER (PARTITION BY Emotion ORDER BY Word_Count DESC) AS Rank_In_Class
    FROM mental_health_eda
)
SELECT 
    Emotion,
    Rank_In_Class,
    Word_Count,
    Char_Length,
    Snippet
FROM RankedPosts
WHERE Rank_In_Class <= 3
ORDER BY Emotion, Rank_In_Class
"""

print("=== Query 1: Top 3 Most Expressive Documents per Category (Window Function) ===")
spark.sql(query_text_window).show(20, truncate=False)


In [ ]:
# Spark SQL Query 2: Ranking Top Destination Accounts by Inbound Traffic partitioned by Verification
query_graph_window = """
WITH DestinationDegree AS (
    SELECT 
        Destination AS Account_ID,
        COALESCE(`Destination Verified`, 'False') AS Is_Verified,
        COUNT(*) AS Inbound_Interactions
    FROM twitter_graph_eda
    WHERE Destination IS NOT NULL
    GROUP BY Destination, COALESCE(`Destination Verified`, 'False')
),
RankedDestinations AS (
    SELECT 
        Account_ID,
        Is_Verified,
        Inbound_Interactions,
        ROW_NUMBER() OVER (
            PARTITION BY Is_Verified 
            ORDER BY Inbound_Interactions DESC
        ) AS Popularity_Rank
    FROM DestinationDegree
)
SELECT 
    Is_Verified,
    Popularity_Rank,
    Account_ID,
    Inbound_Interactions
FROM RankedDestinations
WHERE Popularity_Rank <= 5
ORDER BY Is_Verified DESC, Popularity_Rank ASC
"""

print("=== Query 2: Top Inbound Destination Accounts Partitioned by Verification Status ===")
spark.sql(query_graph_window).show(truncate=False)


## 6. Executive Consulting Takeaways & Architectural Next Steps

### Summary of Findings for Client Stakeholders:
1. **Target Balance (Mental Health):** The dataset spans 6 emotional categories, with `Positive` (23.0%) and `Neutral` (18.6%) representing the largest segments, while `Anxiety` (11.1%) is the smallest. Downstream MLlib models must utilize **Weighted Precision/Recall/F1-Score** to prevent majority class bias.
2. **Text Complexity:** Document length averages ~19 words with a standard deviation of ~11 words, though max lengths exceed 60 words. This confirms that basic word tokenization and TF-IDF / HashingTF will produce compact, dense feature vectors without extreme sparsity.
3. **Network Centrality & Skewness:** The graph exhibits classical **scale-free power-law properties**: a vast majority of accounts broadcast only 1–2 interactions, while a small elite core of hub accounts receive thousands of inbound interactions.
4. **Data Veracity & Schema Integrity:** Destination attributes in the graph exhibit ~8.5% null rates. Explicit schemas and vertex deduplication are strictly required to ensure GraphFrames stability.

### Transition to Notebook 2 (Data Engineering & Preprocessing):
* Construct a **Multi-Hop Medallion Architecture (Bronze -> Silver -> Gold)**.
* Standardize graph tables into compliant **`vertices_df`** (`id`, attributes) and **`edges_df`** (`src`, `dst`, attributes).
* Clean social text with unified regex and fit a **`StringIndexer`** for MLlib classification.
